# 04 — Limpeza dos Informes Semanais de Arboviroses (2025)

**Projeto Aplicado II — Guilherme Cunha Oliveira (RA 10754621) e Gabriel Lucca Simionato (RA 10751300)**

Notebook nas 10 seções do *Plano de Limpeza das Bases*. PDFs preservados em `db/bases_brutas/informes/`; corpus em `db/bases_tratadas/informes/`.

## 1. Objetivo

Construir um **corpus textual rastreável** a partir dos 17 informes em PDF, seguindo a seção 6 do plano: extração de texto por página, remoção de cabeçalhos e rodapés, identificação da página, seleção da seção de dengue, normalização de espaços e quebras. A saída tem a estrutura recomendada — `informe`, `pagina`, `texto_original`, `texto_limpo` — acrescida da marcação da seção de dengue. Nenhum valor é transformado em variável clínica aqui, e os acumulados de informes sucessivos não são somados: cada informe é lido no seu próprio período.

In [1]:
# ============================================================
# 1. OBJETIVO
# ============================================================
ENTRADA_PREVISTA = "db/bases_brutas/informes/*.pdf"
SAIDA_PREVISTA   = "db/bases_tratadas/informes/informes_corpus_2025.csv"

## 2. Bibliotecas

`pdfplumber` para extrair texto por página mantendo a posição das linhas (o que permite identificar cabeçalhos e rodapés pela altura na página), `re` e `unicodedata` para as regras textuais, pandas para organizar o corpus.

In [2]:
# ============================================================
# 2. BIBLIOTECAS
# ============================================================
import re, json, hashlib, unicodedata, time, warnings
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import pdfplumber
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160, "display.max_colwidth", 120)
print("pdfplumber", pdfplumber.__version__, "| pandas", pd.__version__)

pdfplumber 0.11.9 | pandas 3.0.2


## 3. Caminhos

In [3]:
# ============================================================
# 3. CAMINHOS
# ============================================================
RAIZ = Path("..").resolve()
BRUTA, TRATADA = RAIZ/"db/bases_brutas/informes", RAIZ/"db/bases_tratadas/informes"
TRATADA.mkdir(parents=True, exist_ok=True)
ARQ_CORPUS   = TRATADA/"informes_corpus_2025.csv"
ARQ_CABEC    = TRATADA/"informes_cabecalhos_rodapes_removidos.csv"
ARQ_SECOES   = TRATADA/"informes_secoes_detectadas.csv"
ARQ_LOG      = TRATADA/"informes_limpeza_log.json"
PDFS = sorted(BRUTA.glob("*.pdf"))
LOG = {"entrada": str(BRUTA.relative_to(RAIZ)), "saida": str(ARQ_CORPUS.relative_to(RAIZ)), "n_informes": len(PDFS), "etapas": {}}
print(f"{len(PDFS)} informe(s) encontrado(s) em {BRUTA.relative_to(RAIZ)}")
for p in PDFS: print("  ", p.name)
if not PDFS: print("\n>>> Coloque os 17 PDFs em db/bases_brutas/informes/ e execute novamente. As seções seguintes rodam sem erro e não gravam corpus vazio.")

17 informe(s) encontrado(s) em db/bases_brutas/informes
   Informe Semanal nº 01 - COE Dengue e outras Arboviroses - SE 1 a 4  - 27 de janeiro.pdf
   Informe Semanal nº 02 - COE Dengue e outras Arboviroses - SE 1 a 8 - 24 de fevereiro.pdf
   Informe Semanal nº 03 - COE Dengue e outras Arboviroses - SE 1 a 9 - 08 de março.pdf
   Informe Semanal nº 04 - COE Dengue e outras Arboviroses - SE 1 a 10.pdf
   Informe Semanal nº 05 - COE Dengue e outras Arboviroses - SE 1 a 11.pdf
   Informe Semanal nº 06 - COE Dengue e outras Arboviroses - SE 1 a 12.pdf
   Informe Semanal nº 07 - COE Dengue e outras Arboviroses - SE 1 a 13.pdf
   Informe Semanal nº 08 - COE Dengue e outras Arboviroses - SE 1 a 13.pdf
   Informe Semanal nº 09 - COE Dengue e outras Arboviroses - SE 1 a 15.pdf
   Informe Semanal nº 10 - COE Dengue e outras Arboviroses - SE 1 a 16.pdf
   Informe Semanal nº 11 - COE Dengue e outras Arboviroses - SE 1 a 17 - 13 de maio.pdf
   Informe Semanal nº 12 - COE Dengue e outras Arboviroses -

## 4. Carregamento

Cada PDF é aberto em modo leitura; para cada página guardamos o texto bruto e as linhas com sua posição vertical (`top`, em fração da altura da página), o tamanho de fonte e se estão em negrito — a posição serve à regra de cabeçalho e rodapé, a fonte à regra de títulos (seção 6). Também extraímos os metadados que identificam o período de cada informe: número, semanas epidemiológicas cobertas (SE inicial e final) e data de publicação, lidos da capa e do cabeçalho. A identidade de cada arquivo (SHA-256) é registrada.

In [4]:
# ============================================================
# 4. CARREGAMENTO
# ============================================================
def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for parte in iter(lambda: f.read(1 << 24), b""): h.update(parte)
    return h.hexdigest()

def carregar_pdf(caminho):
    paginas = []
    with pdfplumber.open(caminho) as pdf:
        for i, pg in enumerate(pdf.pages, start=1):
            altura = float(pg.height) or 1.0
            try:
                linhas = [{"texto": l["text"], "top": l["top"]/altura,
                           "size": max(c["size"] for c in l["chars"]), "bold": any("Bold" in c["fontname"] for c in l["chars"])}
                          for l in pg.extract_text_lines(strip=True, return_chars=True)]
            except Exception:
                linhas = [{"texto": t, "top": np.nan, "size": np.nan, "bold": False} for t in (pg.extract_text() or "").splitlines()]
            paginas.append({"pagina": i, "texto_original": pg.extract_text() or "", "linhas": linhas, "n_chars": len(pg.chars)})
    return paginas

RE_NUM_INFORME = re.compile(r"n[º°o\.]*\s*(\d{1,2})", re.I)
RE_SE = re.compile(r"SE\s*(\d{1,2})\s*a\s*(?:SE\s*)?(\d{1,2})\s*/\s*(20\d\d)", re.I)
RE_DATA = re.compile(r"\((\d{2}/\d{2}/\d{4})\)")
def metadados(nome, pags):
    texto_ini = " ".join(p["texto_original"] for p in pags[:2])
    num = RE_NUM_INFORME.search(nome); se = RE_SE.search(texto_ini); dt = RE_DATA.search(texto_ini)
    return {"informe_num": int(num.group(1)) if num else None, "se_inicio": int(se.group(1)) if se else None, "se_fim": int(se.group(2)) if se else None,
            "ano_se": int(se.group(3)) if se else None, "data_informe": pd.to_datetime(dt.group(1), dayfirst=True).date().isoformat() if dt else None}

t0 = time.time(); docs = {}
meta = {}
for p in PDFS:
    docs[p.name] = carregar_pdf(p); meta[p.name] = metadados(p.name, docs[p.name])
    LOG["etapas"].setdefault("identidade", {})[p.name] = {"tamanho_KB": round(p.stat().st_size/1e3), "sha256": sha256(p), "paginas": len(docs[p.name]), **meta[p.name]}
print(f"{len(docs)} documento(s), {sum(len(v) for v in docs.values())} páginas, {time.time()-t0:.0f} s")
meta = pd.DataFrame(meta).T.sort_values("informe_num") if meta else pd.DataFrame()
meta

17 documento(s), 343 páginas, 31 s


,informe_num,se_inicio,se_fim,ano_se,data_informe
Informe Semanal nº 01 - COE Dengue e outras Arboviroses - SE 1 a 4 - 27 de janeiro.pdf,1,1,4,2025,2025-01-27
Informe Semanal nº 02 - COE Dengue e outras Arboviroses - SE 1 a 8 - 24 de fevereiro.pdf,2,1,8,2025,2025-02-24
Informe Semanal nº 03 - COE Dengue e outras Arboviroses - SE 1 a 9 - 08 de março.pdf,3,1,9,2025,2025-03-08
Informe Semanal nº 04 - COE Dengue e outras Arboviroses - SE 1 a 10.pdf,4,1,10,2025,2025-03-10
Informe Semanal nº 05 - COE Dengue e outras Arboviroses - SE 1 a 11.pdf,5,1,11,2025,2025-03-17
Informe Semanal nº 06 - COE Dengue e outras Arboviroses - SE 1 a 12.pdf,6,1,12,2025,2025-03-24
Informe Semanal nº 07 - COE Dengue e outras Arboviroses - SE 1 a 13.pdf,7,1,13,2025,2025-03-31
Informe Semanal nº 08 - COE Dengue e outras Arboviroses - SE 1 a 13.pdf,8,1,14,2025,2025-04-07
Informe Semanal nº 09 - COE Dengue e outras Arboviroses - SE 1 a 15.pdf,9,1,15,2025,2025-04-14
Informe Semanal nº 10 - COE Dengue e outras Arboviroses - SE 1 a 16.pdf,10,1,16,2025,2025-04-22


## 5. Diagnóstico inicial

Para cada informe: número de páginas, páginas sem camada de texto (PDF digitalizado exigiria OCR, limite registrado no mapeamento), tamanho do texto por página e as linhas que se repetem em muitas páginas — candidatas a cabeçalho e rodapé. Também levantamos os tamanhos de fonte e os títulos em negrito, que fundamentam a regra de seleção da seção de dengue: nos informes do COE, o corpo está em 14 pt e cada doença (Dengue, Chikungunya, Zika, Oropouche, Febre Amarela) abre uma seção com título em negrito de 20 pt; subtítulos como "Diagrama de controle" têm 16 pt; e as tabelas anexas do fim têm título em negrito de 12 pt no alto da página, mencionando a doença.

In [5]:
# ============================================================
# 5. DIAGNÓSTICO INICIAL
# ============================================================
def normalizar_chave(t):   # chave para comparar linhas repetidas: sem acento, minúsculas, sem dígitos (números de página/semana variam)
    t = unicodedata.normalize("NFKD", t).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", re.sub(r"\d+", "#", t.lower())).strip()

diag = []
repetidas = {}
for nome, pags in docs.items():
    n = len(pags); sem_texto = sum(1 for p in pags if p["n_chars"] == 0)
    chars = [len(p["texto_original"]) for p in pags]
    cont = Counter(normalizar_chave(l["texto"]) for p in pags for l in p["linhas"] if l["texto"].strip())
    rep = {k: v for k, v in cont.items() if n >= 3 and v >= max(3, 0.5*n)}
    repetidas[nome] = rep
    diag.append({"informe": nome, "paginas": n, "paginas_sem_texto": sem_texto, "chars_medio_pagina": int(np.mean(chars)) if chars else 0,
                 "chars_min": min(chars) if chars else 0, "linhas_repetidas_50%": len(rep)})
diag = pd.DataFrame(diag)
print(diag.to_string() if len(diag) else "sem documentos")

                                                                                     informe  paginas  paginas_sem_texto  chars_medio_pagina  chars_min  linhas_repetidas_50%
0    Informe Semanal nº 01 - COE Dengue e outras Arboviroses - SE 1 a 4  - 27 de janeiro.pdf       21                  0                1027         43                     3
1   Informe Semanal nº 02 - COE Dengue e outras Arboviroses - SE 1 a 8 - 24 de fevereiro.pdf       22                  0                1131         45                     7
2       Informe Semanal nº 03 - COE Dengue e outras Arboviroses - SE 1 a 9 - 08 de março.pdf       20                  0                 989         41                     7
3                    Informe Semanal nº 04 - COE Dengue e outras Arboviroses - SE 1 a 10.pdf       20                  0                1029         41                     7
4                    Informe Semanal nº 05 - COE Dengue e outras Arboviroses - SE 1 a 11.pdf       20                  0          

In [6]:
for nome, rep in list(repetidas.items())[:3]:
    print(f"\n== {nome}: linhas repetidas em ≥50% das páginas (candidatas a cabeçalho/rodapé)")
    for k, v in sorted(rep.items(), key=lambda kv: -kv[1])[:8]: print(f"   [{v:>3} págs] {k[:100]}")


== Informe Semanal nº 01 - COE Dengue e outras Arboviroses - SE 1 a 4  - 27 de janeiro.pdf: linhas repetidas em ≥50% das páginas (candidatas a cabeçalho/rodapé)
   [ 26 págs] #
   [ 20 págs] informe semanal no #
   [ 20 págs] se # a #/# (#/#/#)

== Informe Semanal nº 02 - COE Dengue e outras Arboviroses - SE 1 a 8 - 24 de fevereiro.pdf: linhas repetidas em ≥50% das páginas (candidatas a cabeçalho/rodapé)
   [ 62 págs] # # # #
   [ 42 págs] # # #,# #,#
   [ 29 págs] # # #,# #,# # #
   [ 23 págs] #
   [ 20 págs] informe semanal no #
   [ 20 págs] se # a #/# (#/#/#)
   [ 14 págs] #.# #.# #,# #,#

== Informe Semanal nº 03 - COE Dengue e outras Arboviroses - SE 1 a 9 - 08 de março.pdf: linhas repetidas em ≥50% das páginas (candidatas a cabeçalho/rodapé)
   [ 60 págs] # # # #
   [ 38 págs] # # #,# #,#
   [ 29 págs] # # #,# #,# # #
   [ 21 págs] #
   [ 19 págs] informe semanal no #
   [ 19 págs] se # a #/# (#/#/#)
   [ 16 págs] #.# #.# #,# #,#


In [7]:
# tamanhos de fonte por documento: o corpo dos informes está em 14 pt e os títulos de seção em negrito ≥ 16 pt
fontes = Counter()
for nome, pags in docs.items():
    for p in pags:
        for l in p["linhas"]:
            if l["size"] == l["size"]: fontes[(round(l["size"]), l["bold"])] += 1
print("combinações (tamanho, negrito) mais frequentes nas linhas:", fontes.most_common(8))
titulos = Counter()
for nome, pags in docs.items():
    for p in pags:
        for l in p["linhas"]:
            if l["bold"] and l["size"] >= 15.5 and len(l["texto"].split()) <= 8 and not re.search(r"\d", l["texto"]): titulos[l["texto"].strip()] += 1
print("\ntítulos em negrito ≥ 16 pt (frequência nos", len(docs), "informes):")
for t, n in titulos.most_common(20): print(f"  {n:3d} | {t}")

combinações (tamanho, negrito) mais frequentes nas linhas: [((10, False), 4854), ((12, False), 1819), ((14, False), 1476), ((10, True), 1247), ((12, True), 815), ((16, True), 398), ((11, False), 397), ((9, False), 257)]

títulos em negrito ≥ 16 pt (frequência nos 17 informes):
   50 | Febre Amarela
   34 | Dengue
   34 | Coeficiente de Incidência e óbitos
   34 | Chikungunya
   34 | Zika
   34 | Oropouche
   33 | Diagrama de controle
   20 | Situação Epidemiológica
   17 | Introdução
   17 | Ações realizadas
   17 | Insumos distribuídos
   15 | Sorologia Reações
   15 | Biologia Molecular ZDC Reações
   15 | Biologia Molecular OROV/MAYV Reações
   15 | Larvicida Kg
   15 | Adulticida para PE Kg
   15 | Adulticida para UBV L
   10 | Biologia Molecular Febre Amarela Reações
    2 | Preparação
    2 | Resposta


## 6. Regras de limpeza

- **R1 — Extração por página.** Texto extraído com `pdfplumber`, uma linha do corpus por página; `texto_original` é o texto tal como extraído.
- **R2 — Cabeçalhos e rodapés.** Uma linha é removida quando (a) sua chave normalizada (sem acentos, minúsculas, dígitos substituídos por `#`) se repete em pelo menos 50% das páginas do mesmo informe e está no topo (`top` < 12%) ou na base (`top` > 88%) da página, ou (b) é apenas um número e está na borda da página (número de página). Números soltos no meio da página — os valores dos infográficos — são preservados. As linhas removidas são gravadas em `informes_cabecalhos_rodapes_removidos.csv`, com informe e página, para conferência.
- **R3 — Página e período.** Toda linha do corpus carrega `informe` (nome do arquivo), `informe_num`, `se_inicio`, `se_fim`, `data_informe` e `pagina` (1-based), o que garante o retorno ao documento de origem e respeita o período de cada informe.
- **R4 — Seção de dengue.** A regra usa a tipografia dos informes. Um **título de seção** é uma linha em negrito com fonte ≥ 16 pt e até 8 palavras; um **título de tabela anexa** é uma linha em negrito de 12 a 15 pt no alto da página (`top` ≤ 12%) depois do cabeçalho. A seção de dengue **abre** em um título (de seção ou de tabela) que contém "dengue" e **fecha** no próximo título que nomeia outra doença (chikungunya, zika, oropouche, febre amarela, mayaro) ou outra parte do informe (introdução, ações realizadas, insumos, vacinação, referências). Subtítulos internos ("Diagrama de controle", "Coeficiente de incidência e óbitos") não mudam o estado. Quando o PDF não traz informação de fonte, vale a regra anterior por aparência da linha. O texto da seção fica em `texto_dengue`, `secao_dengue = 1` marca as páginas com conteúdo, `tipo_secao_dengue` distingue **narrativa** (aberta por título de seção: texto, diagrama, infográfico) de **tabela** (aberta por título de tabela anexa), e os títulos detectados vão para `informes_secoes_detectadas.csv` para **revisão manual** — a regra é rastreável, não infalível.
- **R5 — Normalização.** Em `texto_limpo`: união de palavras hifenizadas na quebra de linha, remoção de caracteres de controle, espaços e quebras múltiplas reduzidos a um espaço; mantêm-se maiúsculas, acentos e pontuação (a análise posterior decide o que retirar).
- **R6 — O que não se faz.** Nenhum número é somado entre informes; nenhuma menção vira variável clínica; páginas sem camada de texto são marcadas (`sem_texto = 1`) e não inventadas.

In [8]:
# ============================================================
# 6. REGRAS DE LIMPEZA
# ============================================================
TOPO, BASE = 0.12, 0.88
RE_NUM_PAG = re.compile(r"^\s*(p[áa]g(ina)?\.?\s*)?\d{1,3}(\s*(/|de)\s*\d{1,3})?\s*$", re.I)
RE_INICIO_DENGUE = re.compile(r"\bdengue\b", re.I)
RE_FIM_DENGUE    = re.compile(r"\b(chikungunya|zika|febre amarela|oropouche|mayaro|introdu[cç][aã]o|a[cç][oõ]es realizadas|insumos|vacina[cç][aã]o|refer[eê]ncias)\b", re.I)
RE_TITULO = re.compile(r"^\s*(\d+(\.\d+)*\.?\s+)?[A-ZÁÉÍÓÚÂÊÔÃÕÇ][^.]{2,80}$")

def e_cabecalho_rodape(linha, rep_doc):
    t = linha["texto"].strip(); top = linha["top"]
    if not t: return True
    borda = (top != top) or top < TOPO or top > BASE                       # sem posição, ou no topo/base da página
    if RE_NUM_PAG.match(t): return borda                                   # número solto só é nº de página se estiver na borda
    return normalizar_chave(t) in rep_doc and borda

def e_titulo(linha):
    t = linha["texto"].strip(); n = len(t.split())
    if linha["size"] == linha["size"]:                                   # há informação de fonte
        secao  = linha["bold"] and linha["size"] >= 15.5 and n <= 8
        tabela = linha["bold"] and 11.5 <= linha["size"] < 15.5 and linha["top"] <= TOPO and n <= 20
        return secao or tabela
    return bool(RE_TITULO.match(t)) and n <= 12                          # sem fonte: aparência da linha

def limpar_texto(t):
    t = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f]", " ", t)
    t = re.sub(r"(\w)-\n(\w)", r"\1\2", t)            # hifenização na quebra de linha
    t = re.sub(r"\s*\n\s*", " ", t)
    return re.sub(r"\s+", " ", t).strip()

## 7. Aplicação

In [9]:
# ============================================================
# 7. APLICAÇÃO
# ============================================================
corpus, removidas, secoes = [], [], []
for nome, pags in docs.items():
    rep_doc = repetidas.get(nome, {}); em_dengue = False; tipo_atual = "narrativa"
    for p in pags:
        mantidas, dengue_linhas = [], []; tipos_pag = set()
        for l in p["linhas"]:
            t = l["texto"].strip()
            if e_cabecalho_rodape(l, rep_doc):
                if t: removidas.append({"informe": nome, "pagina": p["pagina"], "linha": t}); 
                continue
            if e_titulo(l):
                if RE_INICIO_DENGUE.search(t):
                    em_dengue = True; tipo_atual = "tabela" if (l["size"] == l["size"] and l["size"] < 15.5) else "narrativa"
                    secoes.append({"informe": nome, "pagina": p["pagina"], "titulo": t, "evento": "inicio_dengue", "tipo": tipo_atual})
                elif em_dengue and RE_FIM_DENGUE.search(t): em_dengue = False; secoes.append({"informe": nome, "pagina": p["pagina"], "titulo": t, "evento": "fim_dengue", "tipo": tipo_atual})
            mantidas.append(t)
            if em_dengue: dengue_linhas.append(t); tipos_pag.add(tipo_atual)
        corpus.append({"informe": nome, **(meta.loc[nome].to_dict() if len(meta) else {}), "pagina": p["pagina"], "sem_texto": int(p["n_chars"] == 0), "n_caracteres": len(p["texto_original"]),
                       "texto_original": p["texto_original"], "texto_limpo": limpar_texto("\n".join(mantidas)),
                       "secao_dengue": int(bool(dengue_linhas)), "tipo_secao_dengue": "/".join(sorted(tipos_pag)) if tipos_pag else "",
                       "texto_dengue": limpar_texto("\n".join(dengue_linhas))})
corpus, removidas, secoes = pd.DataFrame(corpus), pd.DataFrame(removidas), pd.DataFrame(secoes)
if len(corpus): corpus = corpus.sort_values(["informe_num","pagina"]).reset_index(drop=True)
print(f"corpus: {len(corpus)} páginas | linhas de cabeçalho/rodapé removidas: {len(removidas)} | títulos de seção detectados: {len(secoes)}")
LOG["etapas"]["aplicacao"] = {"paginas": int(len(corpus)), "linhas_removidas": int(len(removidas)), "titulos_secao": int(len(secoes)),
                              "paginas_com_secao_dengue": int(corpus.secao_dengue.sum()) if len(corpus) else 0}

corpus: 343 páginas | linhas de cabeçalho/rodapé removidas: 976 | títulos de seção detectados: 138


## 8. Validação

Cada página do corpus tem informe e número de página; o texto limpo não é maior que o original; toda página com `secao_dengue = 1` pertence a um informe em que um título de dengue foi detectado; e conferimos, por amostragem, as linhas removidas e o início de cada seção de dengue.

In [10]:
# ============================================================
# 8. VALIDAÇÃO
# ============================================================
if len(corpus):
    checks = {
     "informe e pagina preenchidos": bool(corpus[["informe","pagina"]].notna().all().all()),
     "(informe, pagina) unico": bool(not corpus.duplicated(["informe","pagina"]).any()),
     "texto_limpo <= texto_original": bool((corpus.texto_limpo.str.len() <= corpus.texto_original.str.len() + 5).all()),
     "secao_dengue so em informes com titulo de dengue": bool(set(corpus.loc[corpus.secao_dengue == 1, "informe"]) <= set(secoes.loc[secoes.evento == "inicio_dengue", "informe"]) if len(secoes) else corpus.secao_dengue.sum() == 0),
     "paginas sem texto marcadas": bool(((corpus.n_caracteres == 0) == (corpus.sem_texto == 1)).all()),
    }
    print(pd.Series(checks).to_string()); assert all(checks.values()); LOG["etapas"]["validacao"] = checks
    print("\npáginas por informe com seção de dengue:"); print(corpus.groupby("informe_num").agg(se=("se_fim","first"), paginas=("pagina","size"), com_dengue=("secao_dengue","sum"), narrativa=("tipo_secao_dengue", lambda s: (s == "narrativa").sum()), tabela=("tipo_secao_dengue", lambda s: (s == "tabela").sum())).to_string())
    print("\namostra de linhas removidas:"); print(removidas.sample(min(10, len(removidas)), random_state=42).to_string() if len(removidas) else "nenhuma")
    print("\ninício das seções de dengue detectadas:"); print(secoes[secoes.evento == "inicio_dengue"].to_string() if len(secoes) else "nenhuma")
else:
    print("nada a validar: nenhum PDF na camada bruta")

informe e pagina preenchidos                        True
(informe, pagina) unico                             True
texto_limpo <= texto_original                       True
secao_dengue so em informes com titulo de dengue    True
paginas sem texto marcadas                          True

páginas por informe com seção de dengue:


             se  paginas  com_dengue  narrativa  tabela
informe_num                                            
1             4       21           8          5       3
2             8       22           7          4       3
3             9       20           7          4       3
4            10       20           7          4       3
5            11       20           7          4       3
6            12       20           7          4       3
7            13       20           7          4       3
8            14       20           7          4       3
9            15       20           7          4       3
10           16       20           7          4       3
11           17       20           7          4       3
12           18       20           7          4       3
13           19       20           7          4       3
14           20       20           7          4       3
15           21       20           7          4       3
16           22       20           7          4 

                                                                                      informe  pagina                                                                                   titulo         evento       tipo
0     Informe Semanal nº 01 - COE Dengue e outras Arboviroses - SE 1 a 4  - 27 de janeiro.pdf       2                                                                                   Dengue  inicio_dengue  narrativa
2     Informe Semanal nº 01 - COE Dengue e outras Arboviroses - SE 1 a 4  - 27 de janeiro.pdf       7                                                                                   Dengue  inicio_dengue  narrativa
4     Informe Semanal nº 01 - COE Dengue e outras Arboviroses - SE 1 a 4  - 27 de janeiro.pdf      15                                                                         Vacina de dengue  inicio_dengue  narrativa
5     Informe Semanal nº 01 - COE Dengue e outras Arboviroses - SE 1 a 4  - 27 de janeiro.pdf      15                               

## 9. Exportação

Três arquivos: o corpus (`informes_corpus_2025.csv`, uma linha por página), as linhas removidas como cabeçalho/rodapé e os títulos de seção detectados — os dois últimos existem justamente para a revisão manual exigida pelo mapeamento. Nada é gravado quando não há PDFs.

In [11]:
# ============================================================
# 9. EXPORTAÇÃO
# ============================================================
if len(corpus):
    corpus.to_csv(ARQ_CORPUS, index=False); removidas.to_csv(ARQ_CABEC, index=False); secoes.to_csv(ARQ_SECOES, index=False)
    LOG["etapas"]["exportacao"] = {"linhas": int(len(corpus)), "colunas": list(corpus.columns), "tamanho_KB": round(ARQ_CORPUS.stat().st_size/1e3)}
    print(json.dumps(LOG["etapas"]["exportacao"], indent=2, ensure_ascii=False))
else:
    LOG["etapas"]["exportacao"] = "não executada: nenhum PDF"; print("nenhum corpus gravado")
LOG["executado_em"] = pd.Timestamp.now().strftime("%Y-%m-%d %H:%M")
ARQ_LOG.write_text(json.dumps(LOG, indent=2, ensure_ascii=False, default=str), encoding="utf-8")

{
  "linhas": 343,
  "colunas": [
    "informe",
    "informe_num",
    "se_inicio",
    "se_fim",
    "ano_se",
    "data_informe",
    "pagina",
    "sem_texto",
    "n_caracteres",
    "texto_original",
    "texto_limpo",
    "secao_dengue",
    "tipo_secao_dengue",
    "texto_dengue"
  ],
  "tamanho_KB": 878
}


7031

## 10. Resumo

In [12]:
# ============================================================
# 10. RESUMO
# ============================================================
pd.DataFrame([
    ["informes (PDF) na camada bruta", len(PDFS), "esperados: 17"],
    ["páginas no corpus", len(corpus), "uma linha por página"],
    ["páginas sem camada de texto", int(corpus.sem_texto.sum()) if len(corpus) else 0, "exigiriam OCR"],
    ["linhas removidas (cabeçalho/rodapé/nº página)", len(removidas), "R2, gravadas para conferência"],
    ["títulos de seção detectados", len(secoes), "R4, revisão manual"],
    ["páginas com seção de dengue", int(corpus.secao_dengue.sum()) if len(corpus) else 0, "R4"],
], columns=["item","valor","observação"])

,item,valor,observação
0,informes (PDF) na camada bruta,17,esperados: 17
1,páginas no corpus,343,uma linha por página
2,páginas sem camada de texto,0,exigiriam OCR
3,linhas removidas (cabeçalho/rodapé/nº página),976,"R2, gravadas para conferência"
4,títulos de seção detectados,138,"R4, revisão manual"
5,páginas com seção de dengue,120,R4
